# Decision-making under uncertainty models

#### Decision-making under uncertainty models from Gonzalez-Jimenez (2024)

In [112]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [113]:


# Define Prelec's probability weighting function
def prelec(p_ij, beta_i, alpha_i):
    """
    Prelec's (1998) probability weighting function.
    
    Parameters:
    p_ij (ixj array): Probability value in the range [0, 1].
    alpha_i (float): Parameter that determines the curvature of the weighting function.
    
    Returns:
    float: Weighted probability.
    """
    if p_ij < 0 or p_ij > 1:
        raise ValueError("x must be in the range [0, 1]")
    
    if alpha_i <= 0:
        raise ValueError("alpha must be positive")
    
    return np.exp(-beta_i*(-np.log(p_ij))**alpha_i)

# Consumption utility function
def utility_function(x_ij, gamma_i):
    """
    Utility function for consumption.
    
    Parameters:
    x_ij (float): Consumption value.
    gamma_i (float): Parameter that determines the curvature of the utility function.
    
    Returns:
    float: Utility value.
    """
    if x_ij < 0:
        raise ValueError("c_i must be non-negative")
    
    if gamma_i <= 0:
        raise ValueError("gamma_i must be positive")
    
    return x_ij**(1 - gamma_i)


In [114]:

# Objective function to minimize
def objective_function(df, params):
    """
    Objective function to minimize the difference between observed and predicted choices.
    Input:
    params (list): List of parameters [alpha_i, beta_i, gamma_i].
    data (DataFrame): Data containing columns 'p_ij', 'x_ij', and 'choice'.
    """
    df = pd.DataFrame(df)
    return(df.columns)
    
    beta_i, alpha_i, gamma_i = params
    errors = []
    
    for row in df.itertuples():
        w_p = prelec(row.p_ij, beta_i, alpha_i)
        u_x = utility_function(row.x_ij, gamma_i)
        true_value = w_p * u_x
        error = (true_value - row.choice) ** 2
        errors.append(error)

    total_mse = np.sum(errors)
    
    return np.mean(errors)


## Test data

In [115]:

# Example data: 
data = {'p_ij': [0.25, 0.5, 0.75],'x_ij': [10, 20, 30],'choice': [1, 0, 1]}
df = pd.DataFrame(data)

errors = []
# Initial parameters
beta_i = 0.8 
alpha_i = 1.0
gamma_i = 0.5

for row in df.itertuples():
    w_p = prelec(row.p_ij, beta_i, alpha_i)
    u_x = utility_function(row.x_ij, gamma_i)
    true_value = w_p * u_x
    error = (true_value - row.choice) ** 2
    errors.append(error)

print(np.mean(errors))

5.943327925072482


In [116]:
minimized_params = minimize(
    objective_function,
    x0=[beta_i, alpha_i, gamma_i],
    args=(df,),
    method='Nelder-Mead',
    options={'disp': True}
)
print("Optimized parameters:", minimized_params.x)
print("Objective function value:", minimized_params.fun)
# Print the final errors
print("Final errors:", errors)
# Print the final parameters
print("Final parameters: beta_i =", beta_i, ", alpha_i =", alpha_i, ", gamma_i =", gamma_i)


Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 10
         Function evaluations: 49
Optimized parameters: [0.8 1.  0.5]
Objective function value: 0.0
Final errors: [np.float64(0.0018630097938678437), np.float64(6.597539553864474), np.float64(11.230581211559105)]
Final parameters: beta_i = 0.8 , alpha_i = 1.0 , gamma_i = 0.5


In [ ]:
# Example of using the estimated parameters
p_example = 0.6
x_example = 25

w_p_example = prelec(p_example, estimated_beta, estimated_alpha)
u_x_example = utility_function(x_example, estimated_gamma)

print(f"Weighted probability (w(p)) for p={p_example}: {w_p_example}")
print(f"Consumption utility (u(x)) for x={x_example}: {u_x_example}")